# Determining the Ideal Machine Type for Gemma 4 31B serving on Vertex AI Endpoints

This notebook guides you through benchmarking and finding the optimal machine type and accelerator configuration for deploying LLMs (specifically **Gemma 4 31B IT**) on a Vertex AI Online Prediction Endpoint.

Unlike traditional predictive models, Large Language Models (LLMs) served via engines like vLLM have unique latency profiles characterized by:
- **Time to First Token (TTFT)**: The latency to process the input prompt and return the first token. This is prompt-size dependent.
- **Inter-Token Latency (ITL)**: The time between subsequent generated tokens.
- **Output Tokens Per Second (TPS)**: The rate at which the model generates text.
- **Aggregate Throughput**: The total number of tokens (input + output) processed by the endpoint per second under concurrency.

This notebook implements an asynchronous benchmarking client that streams responses from a Vertex AI Endpoint to calculate these metrics under configurable levels of concurrent request traffic. It then helps you evaluate cost-effectiveness (Tokens per Dollar) across different machine configurations (e.g., L4 vs. H100 vs. TPU).

### Install Dependencies

Install the required Python packages for accessing Vertex AI, managing asynchronous request concurrency, and visualizing performance metrics.

In [ ]:
%pip install -U google-cloud-aiplatform google-genai python-dotenv nest-asyncio aiohttp matplotlib numpy pandas tabulate

### Setup and Configurations

Set your Google Cloud Project ID, the region where you want to run your endpoints (or where they are already deployed), and optionally authenticate if you're not running in a pre-authenticated environment like Google Cloud Vertex AI Workbench.

In [ ]:
import os
import sys
from google.cloud import aiplatform

# Load dotenv if available locally
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Define your project ID and region
# (Fallback to environment variables if set in .env)
PROJECT_ID = os.environ.get("PROJECT_ID", "")  # @param {type:"string"}
REGION = os.environ.get("REGION", "us-central1")  # @param {type:"string"}

# If running locally or on workbench, fallback to gcloud configuration if empty
if not PROJECT_ID:
    import subprocess
    try:
        PROJECT_ID = subprocess.check_output(
            ["gcloud", "config", "get-value", "project"]
        ).decode("utf-8").strip()
    except Exception:
        pass

print(f"Project ID: {PROJECT_ID}")
print(f"Region: {REGION}")

# Initialize Vertex AI SDK
aiplatform.init(project=PROJECT_ID, location=REGION)

# Existing Endpoint Configuration (Fallback to environment variables if set in .env)
EXISTING_ENDPOINT_ID = os.environ.get("EXISTING_ENDPOINT_ID", "")  # @param {type:"string"}

# Model configuration to benchmark (if deploying a new endpoint)
# Select or uncomment the desired model ID:
MODEL_ID = "google/gemma3@gemma-3-1b-it"  # @param {type:"string"}
# MODEL_ID = "google/gemma4@gemma-4-12b-it"
# MODEL_ID = "google/gemma4@gemma-4-31b-it"
# MODEL_ID = "google/gemma3@gemma-3-27b-it"

ENDPOINT_DISPLAY_NAME = "gemma-benchmark-endpoint"  # @param {type:"string"}

### Model Garden Deployer

Define helper functions to deploy models from Vertex AI Model Garden.

In [ ]:
import subprocess
import json
import time

def deploy_model_garden(
    project: str,
    region: str,
    model: str,
    machine_type: str,
    accelerator_type: str,
    accelerator_count: int,
    display_name: str
) -> str:
    """Deploys a Model Garden model using gcloud CLI synchronously and blocks until active."""
    print(f"Submitting deployment request for model {model} (this blocks and waits until deployment is complete)...")
    
    cmd = [
        "gcloud", "ai", "model-garden", "models", "deploy",
        f"--project={project}",
        f"--region={region}",
        f"--model={model}",
        f"--machine-type={machine_type}",
        f"--accelerator-type={accelerator_type}",
        f"--accelerator-count={accelerator_count}",
        f"--endpoint-display-name={display_name}",
        "--format=json"
    ]
    
    # Run synchronously
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0:
        print(f"Deployment failed: {result.stderr}")
        return ""
        
    print("Deployment completed successfully.")
    try:
        deploy_info = json.loads(result.stdout)
        print(f"Deployment metadata: {deploy_info}")
    except Exception:
        print(f"Raw output: {result.stdout}")
        
    # Find the endpoint ID
    endpoints = aiplatform.Endpoint.list(filter=f'display_name="{display_name}"')
    if endpoints:
        endpoint = endpoints[0]
        print(f"Active Endpoint: {endpoint.resource_name} (ID: {endpoint.name})")
        return endpoint.name
    else:
        print("Endpoint ID not found in list.")
        return ""

# Default parameters to deploy a certified Gemma 3 1B IT endpoint (L4 GPU)
MACHINE_TYPE = "g2-standard-12"
ACCELERATOR_TYPE = "NVIDIA_L4"
ACCELERATOR_COUNT = 1

if not EXISTING_ENDPOINT_ID:
    ENDPOINT_ID = deploy_model_garden(
        project=PROJECT_ID,
        region=REGION,
        model=MODEL_ID,
        machine_type=MACHINE_TYPE,
        accelerator_type=ACCELERATOR_TYPE,
        accelerator_count=ACCELERATOR_COUNT,
        display_name=ENDPOINT_DISPLAY_NAME
    )
else:
    ENDPOINT_ID = EXISTING_ENDPOINT_ID
    print(f"Using existing Endpoint ID: {ENDPOINT_ID}")

### Asynchronous Benchmarking Client

To measure the latency of Large Language Models served on Vertex AI Dedicated Endpoints, we build an asynchronous client using `aiohttp`.

#### Key Metrics Measured:
1. **Time to First Token (TTFT)**: $\text{Timestamp of first token received} - \text{Timestamp of request sent}$.
2. **Inter-Token Latency (ITL)**: The average time taken to generate each token after the first token.
3. **Output Tokens Per Second (TPS)**: $\frac{\text{Total Generated Tokens}}{\text{Generation Time}}$.
4. **Successful Requests Ratio**: Percentage of HTTP 200 responses.

The client sends requests to the dedicated DNS endpoint of the Vertex AI Prediction Endpoint. It uses streaming (`"stream": true`) to track precise token timestamps. We approximate the number of generated tokens by counting the characters returned in the streaming chunks and dividing by $4$ (a standard heuristic for English text where 1 token $\approx$ 4 characters).

In [ ]:
import asyncio
import aiohttp
import time
import json
from google.auth import default
from google.auth.transport.requests import Request

# Optional exact tokenizer setup
# To use exact tokenizer, run: %pip install transformers
TOKENIZER = None
USE_EXACT_TOKENIZER = False  # @param {type:"boolean"}

if USE_EXACT_TOKENIZER:
    try:
        from transformers import AutoTokenizer
        TOKENIZER = AutoTokenizer.from_pretrained("google/gemma-4-31b-it")
        print("Loaded Hugging Face Autotokenizer successfully.")
    except Exception as e:
        print(f"Could not load Hugging Face Autotokenizer, falling back to 4-character heuristic: {e}")

def get_auth_token():
    """Retrieves Google Cloud access token."""
    credentials, _ = default()
    credentials.refresh(Request())
    return credentials.token

def count_tokens(text):
    """Counts tokens using tokenizer or 4-character heuristic."""
    if TOKENIZER:
        try:
            return len(TOKENIZER.encode(text))
        except Exception:
            pass
    return max(1, round(len(text) / 4.0))

async def execute_stream_turn(session, url, headers, payload):
    """Executes a single streaming API request and returns detailed token logs."""
    start_time = time.time()
    ttft = None
    first_token_time = None
    completion_time = None
    generated_texts = []
    status_code = 0
    error_msg = ""
    
    try:
        async with session.post(url, headers=headers, json=payload) as response:
            status_code = response.status
            if status_code != 200:
                error_msg = await response.text()
                return False, {"status_code": status_code, "error": error_msg, "latency": time.time() - start_time}
            
            async for line in response.content:
                line_str = line.decode('utf-8').strip()
                if not line_str:
                    continue
                
                if line_str.startswith("data:"):
                    data_content = line_str[5:].strip()
                    if data_content == "[DONE]":
                        break
                        
                    try:
                        chunk_json = json.loads(data_content)
                        choices = chunk_json.get("choices", [])
                        if choices:
                            delta = choices[0].get("delta", {})
                            content = delta.get("content", "")
                            if content:
                                generated_texts.append(content)
                                if ttft is None:
                                    first_token_time = time.time()
                                    ttft = first_token_time - start_time
                    except Exception:
                        pass
            completion_time = time.time()
    except Exception as e:
        return False, {"status_code": 500, "error": str(e), "latency": time.time() - start_time}
        
    total_latency = completion_time - start_time
    full_text = "".join(generated_texts)
    tokens_generated = count_tokens(full_text)
    
    tps = 0
    tpot = None
    if tokens_generated > 0 and first_token_time and completion_time:
        generation_time = completion_time - first_token_time
        if generation_time > 0:
            tps = tokens_generated / generation_time
            tpot = generation_time / tokens_generated
            
    return True, {
        "ttft": ttft,
        "total_latency": total_latency,
        "tokens_generated": tokens_generated,
        "tps": tps,
        "tpot": tpot,
        "full_text": full_text
    }

async def send_agentic_request(session, url, headers, endpoint_id, prompt, max_tokens, request_id, num_turns=2, temperature=0.0):
    """Simulates an N-turn agent loop using alternating plain text messages to maintain serving compatibility."""
    start_total = time.time()
    messages = []
    turn_results = []
    
    for turn in range(1, num_turns + 1):
        if turn == 1:
            messages.append({"role": "user", "content": f"{prompt} (Available tools: weather_lookup)"})
        elif turn == num_turns:
            # Final turn: mock final tool output and ask model to summarize
            messages.append({"role": "user", "content": f"Tool response for turn {turn-1}: {{\"temperature\": 18, \"condition\": \"sunny\"}}"})
        else:
            # Intermediate turns: alternate assistant and user tool calls
            messages.append({"role": "assistant", "content": f"I will call weather_lookup. call: weather_lookup(location='Paris')"})
            messages.append({"role": "user", "content": f"Tool response for turn {turn-1}: {{\"temperature\": 18, \"condition\": \"sunny\"}}"})
            
        payload = {
            "model": endpoint_id,
            "messages": messages,
            "max_tokens": max_tokens,
            "stream": True,
            "temperature": temperature
        }
        
        success, res = await execute_stream_turn(session, url, headers, payload)
        if not success:
            return {
                "request_id": request_id,
                "success": False,
                "error": f"Turn {turn} failed: {res.get('error')}",
                "turn1_ttft": None,
                "final_turn_ttft": None,
                "total_latency": time.time() - start_total,
                "total_tokens": 0,
                "ntpot": 0
            }
            
        turn_results.append(res)
        # Append the assistant's generated response to history for next turn
        messages.append({"role": "assistant", "content": res["full_text"]})
        
    total_latency = time.time() - start_total
    total_tokens = sum([r["tokens_generated"] for r in turn_results])
    
    return {
        "request_id": request_id,
        "success": True,
        "turn1_ttft": turn_results[0]["ttft"],
        "final_turn_ttft": turn_results[-1]["ttft"],
        "turn1_tps": turn_results[0]["tps"],
        "final_turn_tps": turn_results[-1]["tps"],
        "total_latency": total_latency,
        "total_tokens": total_tokens,
        "ntpot": total_latency / max(1, total_tokens)
    }

### Concurrent Load Test Runner

The runner schedules and executes batch jobs for different levels of concurrency. It uses an `asyncio.Semaphore` to maintain the target concurrent request limit while sending a total of $N$ requests.

You can configure:
- **`PROMPT`**: The prompt text to benchmark.
- **`MAX_OUTPUT_TOKENS`**: The maximum output tokens to limit response length.
- **`CONCURRENCY_LEVELS`**: A list of concurrent users to test (e.g. `[1, 2, 4, 8, 16, 32]`).
- **`REQUESTS_PER_RUN`**: Total requests to execute per concurrency level test to gather averages.

Run the cell below to set up the runner logic. It will automatically resolve the dedicated endpoint URL, gather authentication tokens, and run the test suite.

In [ ]:
async def run_concurrency_test(url, headers, endpoint_id, prompts, max_tokens, total_requests, concurrency, num_turns=2, temperature=0.0):
    """Runs a concurrent load test using semaphore for a specific concurrency level."""
    semaphore = asyncio.Semaphore(concurrency)
    
    async def worker(session, req_id):
        current_prompt = prompts[req_id % len(prompts)] if isinstance(prompts, list) else prompts
        async with semaphore:
            return await send_agentic_request(session, url, headers, endpoint_id, current_prompt, max_tokens, req_id, num_turns, temperature)
            
    async with aiohttp.ClientSession() as session:
        tasks = [worker(session, i) for i in range(total_requests)]
        start_suite = time.time()
        results = await asyncio.gather(*tasks)
        end_suite = time.time()
        
    total_time = end_suite - start_suite
    return results, total_time

def process_benchmark_results(results, total_time, concurrency):
    """Processes raw request logs and computes summary statistics."""
    success_results = [r for r in results if r["success"]]
    failed_results = [r for r in results if not r["success"]]
    
    total_reqs = len(results)
    success_rate = (len(success_results) / total_reqs) * 100 if total_reqs > 0 else 0
    
    turn1_ttfts = [r["turn1_ttft"] for r in success_results if r["turn1_ttft"] is not None]
    final_turn_ttfts = [r["final_turn_ttft"] for r in success_results if r["final_turn_ttft"] is not None]
    
    turn1_tps_vals = [r["turn1_tps"] for r in success_results if r.get("turn1_tps") is not None]
    final_turn_tps_vals = [r["final_turn_tps"] for r in success_results if r.get("final_turn_tps") is not None]
    
    ntpot_values = [r["ntpot"] for r in success_results if r.get("ntpot") is not None]
    total_tokens = sum([r["total_tokens"] for r in success_results])
    
    # Calculate statistics
    avg_turn1_ttft = sum(turn1_ttfts) / len(turn1_ttfts) if turn1_ttfts else 0
    avg_final_turn_ttft = sum(final_turn_ttfts) / len(final_turn_ttfts) if final_turn_ttfts else 0
    
    p99_turn1_ttft = np.percentile(turn1_ttfts, 99) if turn1_ttfts else 0
    p99_final_turn_ttft = np.percentile(final_turn_ttfts, 99) if final_turn_ttfts else 0
    
    avg_turn1_tps = sum(turn1_tps_vals) / len(turn1_tps_vals) if turn1_tps_vals else 0
    avg_final_turn_tps = sum(final_turn_tps_vals) / len(final_turn_tps_vals) if final_turn_tps_vals else 0
    
    avg_ntpot = sum(ntpot_values) / len(ntpot_values) if ntpot_values else 0
    agg_tps = total_tokens / total_time if total_time > 0 else 0
    req_throughput = len(success_results) / total_time if total_time > 0 else 0
    
    return {
        "concurrency": concurrency,
        "total_requests": total_reqs,
        "success_rate_percent": success_rate,
        "total_time_seconds": total_time,
        "req_throughput_qps": req_throughput,
        "avg_turn1_ttft_seconds": avg_turn1_ttft,
        "avg_final_turn_ttft_seconds": avg_final_turn_ttft,
        "p99_turn1_ttft_seconds": p99_turn1_ttft,
        "p99_final_turn_ttft_seconds": p99_final_turn_ttft,
        "avg_turn1_tps": avg_turn1_tps,
        "avg_final_turn_tps": avg_final_turn_tps,
        "avg_ntpot_seconds": avg_ntpot,
        "aggregate_tps": agg_tps,
        "total_tokens_generated": total_tokens
    }

async def run_benchmark_suite(endpoint_id, prompt, max_tokens, concurrency_levels, requests_per_run=30, num_turns=2, temperature=0.0):
    """Executes full benchmark suite across multiple concurrency levels."""
    print("Resolving Endpoint Dedicated DNS...")
    endpoint = aiplatform.Endpoint(endpoint_name=endpoint_id)
    dedicated_dns = endpoint.dedicated_endpoint_dns
    
    if not dedicated_dns:
        raise ValueError("Dedicated DNS not found for this endpoint. Make sure it is fully deployed.")
        
    url = f"https://{dedicated_dns}/v1beta1/projects/{PROJECT_ID}/locations/{REGION}/endpoints/{endpoint_id}/chat/completions"
    print(f"Benchmark URL: {url}")
    
    token = get_auth_token()
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    import numpy as np # Import locally for use in process_benchmark_results
    
    suite_summary = []
    
    for concurrency in concurrency_levels:
        print(f"\n--- Running Benchmark for Concurrency={concurrency} (Total Requests={requests_per_run}) ---")
        results, total_time = await run_concurrency_test(
            url=url,
            headers=headers,
            endpoint_id=endpoint_id,
            prompts=prompt,
            max_tokens=max_tokens,
            total_requests=requests_per_run,
            concurrency=concurrency,
            num_turns=num_turns,
            temperature=temperature
        )
        
        summary = process_benchmark_results(results, total_time, concurrency)
        suite_summary.append(summary)
        
        print(f"Success Rate: {summary['success_rate_percent']:.1f}%")
        print(f"Throughput: {summary['req_throughput_qps']:.2f} req/s")
        print(f"Avg Turn 1 TTFT: {summary['avg_turn1_ttft_seconds']:.3f} s | Avg Final Turn TTFT: {summary['avg_final_turn_ttft_seconds']:.3f} s")
        print(f"Avg Turn 1 TPS: {summary['avg_turn1_tps']:.1f} | Avg Final Turn TPS: {summary['avg_final_turn_tps']:.1f}")
        print(f"Average NTPOT: {summary['avg_ntpot_seconds']:.3f} s/token")
        print(f"Aggregate Output TPS: {summary['aggregate_tps']:.1f} tokens/s")
        
    return suite_summary

### Execution and Cost-Performance Analysis

In this section, we configure the benchmark parameters, execute the runner, and analyze the results.

We also define a cost lookup dictionary for common Vertex AI dedicated prediction configurations (list prices per hour). This allows us to calculate **Tokens per Dollar ($)**, defined as:
$$\text{Tokens per Dollar} = \frac{\text{Aggregate Output TPS} \times 3600 \text{ seconds/hour}}{\text{Hourly Cost of Machine Type}}$$

This metric directly indicates which machine type is the most cost-effective under different levels of concurrency.

---

### LLM Metrics & GPU Saturation Detection

#### 1. Normalized Time Per Output Token (NTPOT)
NTPOT is defined as:
$$\text{NTPOT} = \frac{\text{Total Latency}}{\text{Tokens Generated}}$$
Unlike standard Time Per Output Token (TPOT) which only measures the generation phase, **NTPOT** represents the complete user-perceived speed per token, factoring in both prompt processing (TTFT) and token generation. It is measured in **seconds per token**.

#### 2. Detecting GPU Saturation Point
To determine the maximum concurrency your GPU setup can handle before saturating:
- **Request Throughput (QPS) Plateau**: Look at the *Request Throughput (QPS) vs Concurrency* plot. As concurrency increases, QPS will increase linearly. The point where QPS flattens (plateaus) is the **saturation point**. Adding more concurrent users beyond this point does not increase throughput; it only increases queuing delay.
- **Latency Knee-Point**: Look at the *P99 TTFT* and *NTPOT* plots. Latencies remain low and flat at first. The concurrency level where latencies start rising sharply (the "knee" of the curve) represents the onset of GPU and vLLM scheduler queue saturation.

---

#### Typical Configuration Costs (List Prices):
- **4 x NVIDIA L4** (`g2-standard-48`): ~\$4.00 / hour
- **1 x NVIDIA A100 (80GB)** (`a2-ultragpu-1g`): ~\$3.67 / hour
- **1 x NVIDIA H100 (80GB)** (`a3-highgpu-1g`): ~\$6.50 / hour

Let's configure the run parameters and run the benchmark. Make sure `ENDPOINT_ID` is set to your active benchmark endpoint.

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np
from tabulate import tabulate
from IPython.display import display, Markdown

# Benchmark Suite Parameters
PROMPT = "What is the weather in Paris?" # @param {type:"string"}
MAX_OUTPUT_TOKENS = 150 # @param {type:"integer"}

# Default concurrency levels to test
CONCURRENCY_LEVELS = [1, 2, 4, 8, 16, 32] # @param {type:"raw"}
# To scale tests up to 256 concurrent requests to find the absolute GPU saturation point,
# uncomment the line below. (Note: For high concurrencies, also increase REQUESTS_PER_RUN to at least 300).
# CONCURRENCY_LEVELS = [1, 2, 4, 8, 16, 32, 64, 128, 256]

REQUESTS_PER_RUN = 30 # @param {type:"integer"}

# Agent Simulation Settings
NUM_TURNS = 2  # @param {type:"integer"}

# Generation parameters
TEMPERATURE = 0.0 # @param {type:"number"}

# Model configuration under test
CURRENT_MODEL_NAME = "gemma-3-1b-it"  # @param {type:"string"}
CURRENT_MACHINE_NAME = "g2-standard-12 (1x L4)"  # @param {type:"string"}
CURRENT_MACHINE_HOURLY_COST = 0.95  # @param {type:"number"}

HISTORY_FILE = "benchmark_history.json"

def save_benchmark_to_history(config_name, hourly_cost, suite_results):
    """Saves the current benchmark results to the local history file for comparisons."""
    history = {}
    if os.path.exists(HISTORY_FILE):
        try:
            with open(HISTORY_FILE, "r") as f:
                history = json.load(f)
        except Exception:
            pass
            
    history[config_name] = {
        "hourly_cost": hourly_cost,
        "results": suite_results
    }
    
    with open(HISTORY_FILE, "w") as f:
        json.dump(history, f, indent=2)
    print(f"Results saved to history for: {config_name}")

def print_comparative_table():
    """Reads history file and prints a formatted markdown table comparing all runs."""
    if not os.path.exists(HISTORY_FILE):
        return
        
    with open(HISTORY_FILE, "r") as f:
        history = json.load(f)
        
    if not history:
        return
        
    table_data = []
    headers = [
        "Model / Machine Configuration", 
        "Hourly Cost", 
        "Concurrency", 
        "Throughput (QPS)", 
        "P99 T1 TTFT (s)", 
        "P99 TF TTFT (s)", 
        "Avg T1 / TF TPS", 
        "NTPOT (s/tok)", 
        "Tokens / Dollar"
    ]
    
    for config_name, data in history.items():
        results = data["results"]
        hourly_cost = data["hourly_cost"]
        
        for r in results:
            concurrency = r["concurrency"]
            qps = r["req_throughput_qps"]
            p99_t1_ttft = r["p99_turn1_ttft_seconds"]
            p99_tf_ttft = r["p99_final_turn_ttft_seconds"]
            avg_t1_tps = r.get("avg_turn1_tps", 0.0)
            avg_tf_tps = r.get("avg_final_turn_tps", 0.0)
            avg_ntpot = r.get("avg_ntpot_seconds", 0.0)
            agg_tps = r["aggregate_tps"]
            
            # Calculate Tokens per Dollar
            tpd = (agg_tps * 3600) / hourly_cost
            
            table_data.append([
                config_name,
                f"${hourly_cost:.2f}",
                concurrency,
                f"{qps:.2f}",
                f"{p99_t1_ttft:.3f}",
                f"{p99_tf_ttft:.3f}",
                f"{avg_t1_tps:.1f} / {avg_tf_tps:.1f}",
                f"{avg_ntpot:.4f}",
                f"{tpd:.0f}"
            ])
            
    # Render table as a beautiful rich-formatted Markdown element
    display(Markdown("## Comparative Summary Table"))
    display(Markdown(tabulate(table_data, headers=headers, tablefmt="github")))

def plot_benchmark_comparison():
    """Generates comparison plots from the history file."""
    if not os.path.exists(HISTORY_FILE):
        print("No benchmark history found. Run the benchmark suite first.")
        return
        
    with open(HISTORY_FILE, "r") as f:
        history = json.load(f)
        
    if not history:
        print("History is empty.")
        return
        
    # Configure high-quality plotting aesthetics
    plt.rcParams['figure.dpi'] = 120
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['grid.alpha'] = 0.3
    plt.rcParams['grid.color'] = '#cccccc'
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Premium curated color palette
    color_palette = ['#1a73e8', '#ea4335', '#fbbc04', '#34a853', '#9c27b0']
    
    for (config_name, data), color in zip(history.items(), color_palette[:len(history)]):
        results = data["results"]
        hourly_cost = data["hourly_cost"]
        
        concurrencies = [r["concurrency"] for r in results]
        p99_t1_ttfts = [r["p99_turn1_ttft_seconds"] for r in results]
        p99_tf_ttfts = [r["p99_final_turn_ttft_seconds"] for r in results]
        agg_tps = [r["aggregate_tps"] for r in results]
        qps = [r["req_throughput_qps"] for r in results]
        
        # Calculate Tokens per Dollar
        tokens_per_dollar = [(tps * 3600) / hourly_cost for tps in agg_tps]
        
        # Subplot 1: P99 TTFT vs Concurrency (comparing both turns)
        axes[0, 0].plot(concurrencies, p99_t1_ttfts, marker='o', linewidth=2.0, label=f"{config_name} (T1 Initial)", color=color)
        axes[0, 0].plot(concurrencies, p99_tf_ttfts, marker='x', linewidth=2.0, linestyle='--', label=f"{config_name} (TF Final)", color=color)
        axes[0, 0].set_title("P99 TTFT vs Concurrency (Turn 1 vs Final Turn)", fontsize=12, fontweight='bold', pad=10)
        axes[0, 0].set_xlabel("Concurrency", fontsize=10)
        axes[0, 0].set_ylabel("TTFT (seconds)", fontsize=10)
        axes[0, 0].grid(True)
        axes[0, 0].legend(frameon=True, facecolor='#f5f5f5', edgecolor='none')
        
        # Subplot 2: Throughput (QPS) vs Concurrency
        axes[0, 1].plot(concurrencies, qps, marker='s', linewidth=2.0, label=f"{config_name}", color=color)
        axes[0, 1].set_title("Request Throughput (QPS) vs Concurrency", fontsize=12, fontweight='bold', pad=10)
        axes[0, 1].set_xlabel("Concurrency", fontsize=10)
        axes[0, 1].set_ylabel("Requests / Second", fontsize=10)
        axes[0, 1].grid(True)
        axes[0, 1].legend(frameon=True, facecolor='#f5f5f5', edgecolor='none')
        
        # Subplot 3: Aggregate Tokens Per Second vs Concurrency
        axes[1, 0].plot(concurrencies, agg_tps, marker='^', linewidth=2.0, label=f"{config_name}", color=color)
        axes[1, 0].set_title("Aggregate Output Tokens/Sec vs Concurrency", fontsize=12, fontweight='bold', pad=10)
        axes[1, 0].set_xlabel("Concurrency", fontsize=10)
        axes[1, 0].set_ylabel("Tokens / Second", fontsize=10)
        axes[1, 0].grid(True)
        axes[1, 0].legend(frameon=True, facecolor='#f5f5f5', edgecolor='none')
        
        # Subplot 4: Tokens per Dollar vs Concurrency
        axes[1, 1].plot(concurrencies, tokens_per_dollar, marker='d', linewidth=2.0, label=f"{config_name} (${hourly_cost:.2f}/hr)", color=color)
        axes[1, 1].set_title("Cost Efficiency (Tokens per Dollar) vs Concurrency", fontsize=12, fontweight='bold', pad=10)
        axes[1, 1].set_xlabel("Concurrency", fontsize=10)
        axes[1, 1].set_ylabel("Tokens per Dollar ($)", fontsize=10)
        axes[1, 1].grid(True)
        axes[1, 1].legend(frameon=True, facecolor='#f5f5f5', edgecolor='none')
        
    plt.tight_layout()
    plt.show()

# Execution wrapper
async def main():
    # Resolve the active endpoint ID from either config or deployment cell
    endpoint_to_use = EXISTING_ENDPOINT_ID if EXISTING_ENDPOINT_ID else (ENDPOINT_ID if 'ENDPOINT_ID' in globals() and ENDPOINT_ID else "")
    if not endpoint_to_use:
        print("Please configure 'EXISTING_ENDPOINT_ID' or run the deployment cell first.")
        return
        
    results = await run_benchmark_suite(
        endpoint_id=endpoint_to_use,
        prompt=PROMPT,
        max_tokens=MAX_OUTPUT_TOKENS,
        concurrency_levels=CONCURRENCY_LEVELS,
        requests_per_run=REQUESTS_PER_RUN,
        num_turns=NUM_TURNS,
        temperature=TEMPERATURE
    )
    
    config_identifier = f"{CURRENT_MODEL_NAME} on {CURRENT_MACHINE_NAME}"
    save_benchmark_to_history(config_identifier, CURRENT_MACHINE_HOURLY_COST, results)
    
    # Plot results and print summary table
    plot_benchmark_comparison()
    print_comparative_table()

# Execute the benchmarking suite in the notebook environment
import asyncio
import nest_asyncio

# Enable nested loop execution for Jupyter kernels
nest_asyncio.apply()
asyncio.run(main())

### AI-Powered Performance Analysis and Recommendations

Use this cell to invoke **Gemini 3.5 Flash** (via the modern Google GenAI SDK and global Vertex AI endpoint) to analyze the gathered benchmark history, identify performance trade-offs, and recommend the best model/machine configuration for your specific SLA and budget.

In [ ]:
from google import genai
from google.genai import types
import os
import json
from IPython.display import display, Markdown

ANALYSIS_MODEL_ID = "gemini-3.5-flash"  # @param {type:"string"}

def generate_ai_analysis():
    if not os.path.exists(HISTORY_FILE):
        print("No benchmark history found. Run the benchmark suite first to record results.")
        return
        
    with open(HISTORY_FILE, "r") as f:
        history = json.load(f)
        
    if not history:
        print("Benchmark history is empty.")
        return
        
    print(f"Initializing GenAI Client via Vertex AI Global Endpoint to call {ANALYSIS_MODEL_ID}...")
    try:
        # Initialize client to use Vertex AI Global Endpoint with active GCP credentials
        client = genai.Client(
            vertexai=True,
            project=PROJECT_ID,
            location="global"  # Using global endpoint for latest model access
        )
        
        prompt = f"""
You are an expert Cloud AI Infrastructure Architect.
Analyze the following JSON structured benchmark results containing performance metrics of different LLM model configurations and machine types served via vLLM on Vertex AI Endpoints.

Raw Benchmark History:
{json.dumps(history, indent=2)}

Please provide a detailed, executive-ready report structured as follows:

1. **Executive Summary**: A high-level overview of the evaluated configurations and the top performing setup.
2. **Cost-Efficiency Analysis (Tokens per Dollar)**: Identify which model and machine configuration is the most cost-effective, comparing the cost of running each hardware configuration vs the token throughput.
3. **Latency & Saturation Analysis**: Analyze how concurrency levels affect TTFT (Time to First Token) and TPS (Tokens Per Second). Identify the concurrency point where latency starts to degrade or where the GPU saturates.
4. **Prefix Caching & Agentic Efficiency**: Compare the differences between Turn 1 (Initial Prompt) and Final Turn (Prompt + History) latency. Explain how well the serving engine caches the prefix history.
5. **Infrastructure Recommendation**: Provide a concrete recommendation on what configuration Maisa.ai should choose based on their need to host Gemma 4 31B for highly agentic workloads.

Format your response in beautiful, readable Markdown.
"""
        
        print("Requesting report from Gemini...")
        response = client.models.generate_content(
            model=ANALYSIS_MODEL_ID,
            contents=[prompt]
        )
        
        # Display as rich formatted markdown instead of raw text print
        display(Markdown("## Gemini Performance Report & Recommendations\n"))
        display(Markdown(response.text))
        
    except Exception as e:
        print(f"Error calling GenAI API: {e}")
        print("Ensure that you have set up correct authentication and the google-genai package is installed.")

# Run analysis
generate_ai_analysis()

### Automated Resource Cleanup

To avoid ongoing Google Cloud billing charges, you must undeploy all models and delete the endpoint once you are finished benchmarking.

Run the cell below with `CLEANUP_ENDPOINT = True` to automatically clean up all resources created or used during this session.

In [ ]:
CLEANUP_ENDPOINT = True  # @param {type:"boolean"}

if CLEANUP_ENDPOINT:
    endpoint_id_to_clean = EXISTING_ENDPOINT_ID if EXISTING_ENDPOINT_ID else (ENDPOINT_ID if 'ENDPOINT_ID' in globals() else "")
    if endpoint_id_to_clean:
        print(f"Cleaning up Endpoint: {endpoint_id_to_clean} in project {PROJECT_ID} ({REGION})...")
        try:
            endpoint = aiplatform.Endpoint(endpoint_name=endpoint_id_to_clean)
            # List and undeploy all deployed models
            deployed_models = endpoint.list_models()
            for model in deployed_models:
                print(f"Undeploying model: {model.id}...")
                endpoint.undeploy(deployed_model_id=model.id)
            # Delete endpoint
            print(f"Deleting Endpoint: {endpoint_id_to_clean}...")
            endpoint.delete()
            print("Cleanup completed successfully.")
        except Exception as e:
            print(f"Error during cleanup: {e}")
    else:
        print("No Endpoint ID specified for cleanup.")
else:
    print("Cleanup bypassed. Set CLEANUP_ENDPOINT = True to clean up resources.")